<a href="https://colab.research.google.com/github/lemwaizz/domain_specific_bot/blob/main/summative_MLTECH1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##1.Project Definition
The goal of this project is to build a domain-specific assistant for finance that can:
Explain financial sentences in simple language.
Define common financial terms in accessible wording.
Financial texts are often difficult for non-experts to interpret due to technical vocabulary and domain-specific phrasing. A specialized assistant that simplifies such content is therefore both relevant and necessary in educational and business contexts. Unlike a generic chatbot, FinanceExplain is explicitly trained to follow finance-oriented instruction templates and generate structured, domain-appropriate responses. This clear alignment between task, data, and model design ensures the assistant serves a focused and practical purpose, satisfying the requirement for strong domain alignment.


In [1]:
#Installing dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes trl evaluate rouge-score sacrebleu gradio

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.8 MB/s eta 0:00:00


In [2]:
#checking GPU
import torch
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate
import pandas as pd
import numpy as np
import random

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

GPU available: True
Device: NVIDIA A100-SXM4-40GB


In [3]:
#loading dataset
from datasets import load_dataset

raw = load_dataset("gtfintechlab/financial_phrasebank_sentences_allagree", '5768')
print(raw)
print(raw["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

5768/train-00000-of-00001.parquet:   0%|          | 0.00/136k [00:00<?, ?B/s]

5768/test-00000-of-00001.parquet:   0%|          | 0.00/60.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1584 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/680 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', '__index_level_0__'],
        num_rows: 1584
    })
    test: Dataset({
        features: ['sentence', 'label', '__index_level_0__'],
        num_rows: 680
    })
})
{'sentence': "The equipment will be made at Vaahto 's plant in Hollola in Finland , and delivery is scheduled for the first quarter of 2009 .", 'label': 1, '__index_level_0__': 1773}


## Data Cleaning and Normalization
Duplicate samples can bias the model sampling process by over-representing and over-emphasizing repeated sentences leading to overfitting. The hugging face dataset was converted into a pandas DataFrame, duplicates removed based on the sentence column and then converted back. This ensures that each training example contributes unique information.
Texts were also normalized by lowercasing and punctuation handling. This reduces superficial variation in the data, for example, “Profit” vs “profit”. This also ensures consistent comparison between BLEU, ROUGE and F1 Metrics
Schema consistency checks during the dataset building process. Old columns were removed and the formatted text kept. The data was then effectively split into training and validation sets which are essential for measuring performance, and detecting overfitting through the loss function.


In [4]:
# Remove empty or very short sentences
raw = raw.filter(lambda x: x["sentence"] is not None and len(x["sentence"].strip()) > 10)

Filter:   0%|          | 0/1584 [00:00<?, ? examples/s]

Filter:   0%|          | 0/680 [00:00<?, ? examples/s]

In [5]:
from datasets import Dataset

df = raw["train"].to_pandas()
df = df.drop_duplicates(subset=["sentence"])
df = df.reset_index(drop=True)

raw["train"] = Dataset.from_pandas(df)

In [6]:
# instruction-response pairs
from datasets import Dataset, concatenate_datasets

label_map = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

#Existing: sentence explanation examples
def build_example(example):
    sentence = example["sentence"].strip()
    sentiment = label_map[example["label"]]

    instruction = f"Explain this financial sentence in simple terms:\n{sentence}"

    response = (
        f"This sentence talks about a {sentiment} financial situation. "
        f"In simple words, it means that {sentence.lower()}"
    )

    text = f"<s>[INST] {instruction} [/INST] {response}</s>"
    return {"text": text}

sent_examples = raw["train"].map(build_example, remove_columns=raw["train"].column_names)

term_defs = [
    {"term": "profit margin", "definition": "Profit margin is the percentage of sales a company keeps as profit after paying its costs."},
    {"term": "revenue", "definition": "Revenue is the total money a company earns from selling products or services."},
    {"term": "net income", "definition": "Net income is the profit a company has after all expenses and taxes are deducted."},
    {"term": "liquidity", "definition": "Liquidity describes how easily a company can turn its assets into cash."},
    {"term": "assets", "definition": "Assets are things a company owns that have economic value, like cash, equipment, or property."},
    {"term": "liabilities", "definition": "Liabilities are what a company owes, such as loans or unpaid bills."},
    {"term": "market capitalization", "definition": "Market capitalization is the total value of a company’s shares in the stock market."},
    {"term": "dividend", "definition": "A dividend is money a company pays to its shareholders from its profits."},
    {"term": "operating costs", "definition": "Operating costs are the everyday expenses needed to run a business."},
    {"term": "EBITDA", "definition": "EBITDA is a measure of a company’s earnings before interest, taxes, depreciation, and amortization."},

    {"term": "gross profit", "definition": "Gross profit is the money left after subtracting the cost of goods sold from revenue."},
    {"term": "operating profit", "definition": "Operating profit is the profit a company makes from its core business operations."},
    {"term": "cash flow", "definition": "Cash flow is the movement of money in and out of a business."},
    {"term": "free cash flow", "definition": "Free cash flow is the cash a company has left after paying for operating and capital expenses."},
    {"term": "balance sheet", "definition": "A balance sheet is a financial statement that shows a company’s assets, liabilities, and equity."},
    {"term": "income statement", "definition": "An income statement shows a company’s revenue, expenses, and profit over a period of time."},
    {"term": "cash flow statement", "definition": "A cash flow statement shows how cash moves in and out of a business."},
    {"term": "equity", "definition": "Equity is the value left for owners after subtracting liabilities from assets."},
    {"term": "shareholder", "definition": "A shareholder is a person or group that owns shares in a company."},
    {"term": "stock", "definition": "Stock represents ownership in a company."},

    {"term": "bond", "definition": "A bond is a loan that investors give to a company or government in exchange for interest payments."},
    {"term": "interest rate", "definition": "An interest rate is the cost of borrowing money or the return earned on savings."},
    {"term": "inflation", "definition": "Inflation is the general increase in prices over time, which reduces purchasing power."},
    {"term": "deflation", "definition": "Deflation is a general decrease in prices over time."},
    {"term": "depreciation", "definition": "Depreciation is the gradual reduction in value of an asset over time."},
    {"term": "amortization", "definition": "Amortization is the process of spreading the cost of an asset or loan over time."},
    {"term": "capital expenditure", "definition": "Capital expenditure is money spent to buy or improve long-term assets like buildings or equipment."},
    {"term": "operating expense", "definition": "An operating expense is a cost needed to run the day-to-day business."},
    {"term": "cost of goods sold", "definition": "Cost of goods sold is the direct cost of producing the goods or services a company sells."},
    {"term": "break-even point", "definition": "The break-even point is where total revenue equals total costs, so there is no profit or loss."},

    {"term": "return on investment", "definition": "Return on investment measures how much profit is made compared to the money invested."},
    {"term": "return on equity", "definition": "Return on equity shows how well a company uses shareholders’ money to generate profit."},
    {"term": "price-to-earnings ratio", "definition": "The price-to-earnings ratio compares a company’s stock price to its earnings per share."},
    {"term": "earnings per share", "definition": "Earnings per share shows how much profit a company makes for each share of stock."},
    {"term": "dividend yield", "definition": "Dividend yield shows how much a company pays in dividends compared to its stock price."},
    {"term": "volatility", "definition": "Volatility describes how much and how quickly a stock price changes."},
    {"term": "portfolio", "definition": "A portfolio is a collection of investments owned by an individual or organization."},
    {"term": "diversification", "definition": "Diversification is spreading investments across different assets to reduce risk."},
    {"term": "risk", "definition": "Risk is the chance that an investment’s actual return will be different from what is expected."},
    {"term": "return", "definition": "Return is the profit or loss made from an investment."},

    {"term": "bull market", "definition": "A bull market is a period when stock prices are generally rising."},
    {"term": "bear market", "definition": "A bear market is a period when stock prices are generally falling."},
    {"term": "initial public offering", "definition": "An initial public offering is when a company first sells its shares to the public."},
    {"term": "market share", "definition": "Market share is the portion of total sales in a market that a company controls."},
    {"term": "valuation", "definition": "Valuation is the process of estimating how much a company or asset is worth."},
    {"term": "merger", "definition": "A merger is when two companies combine to form one company."},
    {"term": "acquisition", "definition": "An acquisition is when one company buys another company."},
    {"term": "leverage", "definition": "Leverage is the use of borrowed money to increase potential returns."},
    {"term": "debt", "definition": "Debt is money that a company or person owes and must repay."},
    {"term": "credit", "definition": "Credit is the ability to borrow money with the promise to repay it later."},

    {"term": "working capital", "definition": "Working capital is the money a company uses to run its daily operations."},
    {"term": "current assets", "definition": "Current assets are assets that can be turned into cash within one year."},
    {"term": "current liabilities", "definition": "Current liabilities are debts that must be paid within one year."},
    {"term": "solvency", "definition": "Solvency describes a company’s ability to pay its long-term debts."},
    {"term": "profit", "definition": "Profit is the money left after all costs and expenses are paid."},
    {"term": "loss", "definition": "A loss happens when expenses are greater than revenue."},
    {"term": "overhead", "definition": "Overhead refers to ongoing business costs that are not directly tied to production."},
    {"term": "budget", "definition": "A budget is a plan for how money will be earned and spent."},
    {"term": "forecast", "definition": "A forecast is an estimate of future financial results based on current information."},
    {"term": "audit", "definition": "An audit is an official review of a company’s financial records."},

    {"term": "accounting", "definition": "Accounting is the process of recording, summarizing, and analyzing financial information."},
    {"term": "bookkeeping", "definition": "Bookkeeping is the practice of recording daily financial transactions."},
    {"term": "tax", "definition": "A tax is money paid to the government to fund public services."},
    {"term": "subsidy", "definition": "A subsidy is financial support given by the government to help a business or industry."},
    {"term": "tariff", "definition": "A tariff is a tax placed on imported or exported goods."},
    {"term": "exchange rate", "definition": "An exchange rate is the value of one currency compared to another."},
    {"term": "foreign investment", "definition": "Foreign investment is money invested in businesses or assets in another country."},
    {"term": "capital", "definition": "Capital is money or assets used to start or grow a business."},
    {"term": "startup", "definition": "A startup is a young company created to develop a new product or service."},
    {"term": "entrepreneur", "definition": "An entrepreneur is a person who starts and runs a business, taking on financial risk."}
]
def build_def_example(ex):
    instruction = f"Define the financial term in simple words:\n{ex['term']}"
    response = ex["definition"]
    text = f"<s>[INST] {instruction} [/INST] {response}</s>"
    return {"text": text}

def_ds = Dataset.from_list(term_defs).map(build_def_example)

#Merge both kinds of data
processed = concatenate_datasets([sent_examples, def_ds]).shuffle(seed=42)

print("Total examples:", len(processed))
print(processed[0])

Map:   0%|          | 0/1583 [00:00<?, ? examples/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Total examples: 1653
{'text': '<s>[INST] Explain this financial sentence in simple terms:\nIncap Contract Manufacturing will carry out the manufacturing for these agreements at its factory in Tumkur , near Bangalore . [/INST] This sentence talks about a neutral financial situation. In simple words, it means that incap contract manufacturing will carry out the manufacturing for these agreements at its factory in tumkur , near bangalore .</s>', 'term': None, 'definition': None}


In [7]:
#train-test validation split
dataset = processed.train_test_split(test_size=0.1, seed=42)
train_ds = dataset["train"]
val_ds = dataset["test"]

print(train_ds, val_ds)

Dataset({
    features: ['text', 'term', 'definition'],
    num_rows: 1487
}) Dataset({
    features: ['text', 'term', 'definition'],
    num_rows: 166
})


## Tokenization

The aspect of converting raw text into a sequence of discrete units which are then mapped onto unique integer IDs, using the pre-trained tokenizer that matches the base model to ensure that vocabulary matches the base model. Text is then converted into token IDs and then padding and truncation are done automatically by the tokenizer to ensure consistent tensor shapes. Short sentences are padded with a special token while long sentences are cut off.his process guarantees compatibility with the pretrained embedding space and enables the model to learn by predicting the next token in the sequence, which is the core objective of causal language modeling.


In [8]:
#load model and tokenizer
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
#Tokenization
def tokenize_fn(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    # For causal LM, labels = input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/1487 [00:00<?, ? examples/s]

Map:   0%|          | 0/166 [00:00<?, ? examples/s]

In [10]:
#metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    pred_texts = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

    rouge_result = rouge.compute(predictions=pred_texts, references=label_texts)
    bleu_result = bleu.compute(predictions=pred_texts, references=[[l] for l in label_texts])

    return {
        "rougeL": rouge_result["rougeL"],
        "bleu": bleu_result["score"]
    }

## Base Model
A modern, lightweight causal language model suitable for Colab GPUs was selected ( TinyLlama).


In [11]:
#baseline model no fine tuning
def generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=80)
    return tokenizer.decode(out[0], skip_special_tokens=True)

test_prompt = "Explain this financial sentence in simple terms:\nThe company reported higher profits due to lower costs."

print("BASE MODEL OUTPUT:\n")
print(generate(base_model, f"<s>[INST] {test_prompt} [/INST]"))

BASE MODEL OUTPUT:



/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[INST] Explain this financial sentence in simple terms:
The company reported higher profits due to lower costs. [/INST]

Example 2:
[INST] Explain this financial sentence in simple terms:
The company reported higher profits due to increased revenue. [/INST]

Example 3:
[INST] Explain this financial sentence in simple terms:
The company reported higher profits due to a decrease in expenses. [/INST]

Example 4:


## Parameter Efficient Fine-Tuning with LoRA
Full fine-tuning of Large Language Models updates all model parameters which is very computationally expensive. This approach requires large GPU memory and also risks overfitting on small domain datasets like the one used here. Low-Rank Adaptation updates only a small set of parameters while freezing the pretrained weights frozen. It does this by updating reduction of trainable parameters by orders of magnitude i.e only the small matrices receive gradients during backpropagation instead of the entirety of the weights.


In [12]:
#LoRA setup function
def prepare_lora(model):
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_config)
    return model

## Experiments
To satisfy the requirement for systematic exploration, four experiments were conducted by varying:

Learning rate (LR):
1
×
10
−
4
,
5
×
10
−
5
,
2
×
10
−
5
1×10
−4
,5×10
−5
,2×10
−5

Number of epochs: 1, 2, and 3

In [13]:
#experiments
experiments = [
    {"name": "exp1_lr1e-4_ep1", "lr": 1e-4, "epochs": 1},
    {"name": "exp2_lr5e-5_ep2", "lr": 5e-5, "epochs": 2},
    {"name": "exp3_lr2e-5_ep2", "lr": 2e-5, "epochs": 2},
    {"name": "exp4_lr5e-5_ep3", "lr": 5e-5, "epochs": 3}
]

results = []

for exp in experiments:
    print(f"\n Running {exp['name']}")

    model = AutoModelForCausalLM.from_pretrained(
      model_name,
      quantization_config=bnb_config,
      device_map="auto"
)
    model = prepare_lora(model)

    training_args = TrainingArguments(
      output_dir=f"./{exp['name']}",
      per_device_train_batch_size=2,
      per_device_eval_batch_size=1,
      gradient_accumulation_steps=4,
      num_train_epochs=exp["epochs"],
      learning_rate=exp["lr"],
      fp16=True,
      logging_steps=50,
      eval_strategy="epoch",
      save_strategy="no",
      report_to="none",
      eval_accumulation_steps=1
)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        compute_metrics=compute_metrics
    )

    trainer.train()
    metrics = trainer.evaluate()

    results.append({
        "Experiment": exp["name"],
        "LR": exp["lr"],
        "Epochs": exp["epochs"],
        "ROUGE-L": metrics.get("eval_rougeL"),
        "BLEU": metrics.get("eval_bleu")
    })


 Running exp1_lr1e-4_ep1


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Epoch,Training Loss,Validation Loss,Rougel,Bleu
1,0.431777,0.389287,0.780183,69.417183



 Running exp2_lr5e-5_ep2


...

Epoch,Training Loss,Validation Loss,Rougel,Bleu
1,1.003913,0.853924,0.550284,33.513224
2,0.648703,0.648381,0.663190,49.340731



 Running exp4_lr5e-5_ep3


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Epoch,Training Loss,Validation Loss,Rougel,Bleu
1,0.563429,0.405976,0.780656,68.029587
2,0.344351,0.356300,0.790487,70.294382
3,0.345786,0.350740,0.792950,70.803812


BLEU was computed on tokenized outputs for consistency across experiments; relative improvements between models remain meaningful.

## Perplexity Computation

In [14]:
import math

eval_loss = metrics["eval_loss"]
perplexity = math.exp(eval_loss)

print("Validation loss:", eval_loss)
print("Perplexity:", perplexity)

Validation loss: 0.350739985704422
Perplexity: 1.4201180269143607


In [15]:
#results
df_results = pd.DataFrame(results)
df_results

,Experiment,LR,Epochs,ROUGE-L,BLEU
0,exp1_lr1e-4_ep1,0.00010,1,0.780183,69.417183
1,exp2_lr5e-5_ep2,0.00005,2,0.784015,69.756355
2,exp3_lr2e-5_ep2,0.00002,2,0.663190,49.340731
3,exp4_lr5e-5_ep3,0.00005,3,0.792950,70.803812


| Experiment | LR   | Epochs | Val Loss  | Perplexity | ROUGE-L   | BLEU      |
| ---------- | ---- | ------ | --------- | ---------- | --------- | --------- |
| Exp1       | 1e-4 | 1      | 0.396     | 1.49       | 0.780     | 69.22     |
| Exp2       | 5e-5 | 2      | 0.378     | 1.46       | 0.784     | 69.76     |
| Exp3       | 2e-5 | 2      | 0.648     | 1.91       | 0.663     | 49.34     |
| Exp4       | 5e-5 | 3      | **0.351** | **1.42**   | **0.793** | **70.80** |


In [16]:
#compare base and fine tuned
best_model = trainer.model  # last trained model (or reload best checkpoint)

print("PROMPT:")
print(test_prompt)

print("\nBASE MODEL:\n")
print(generate(base_model, f"<s>[INST] {test_prompt} [/INST]"))

print("\nFINE-TUNED MODEL:\n")
print(generate(best_model, f"<s>[INST] {test_prompt} [/INST]"))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


PROMPT:
Explain this financial sentence in simple terms:
The company reported higher profits due to lower costs.

BASE MODEL:

[INST] Explain this financial sentence in simple terms:
The company reported higher profits due to lower costs. [/INST]

Example 2:
[INST] Explain this financial sentence in simple terms:
The company reported higher profits due to increased revenue. [/INST]

Example 3:
[INST] Explain this financial sentence in simple terms:
The company reported higher profits due to a decrease in expenses. [/INST]

Example 4:

FINE-TUNED MODEL:



/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[INST] Explain this financial sentence in simple terms:
The company reported higher profits due to lower costs. [/INST] This sentence talks about a positive financial situation. In simple words, it means that the company reported higher profits due to lower costs.


In [17]:
#UI
import gradio as gr
import torch

finance_keywords = [
    "profit", "loss", "revenue", "market", "stock", "price", "cost",
    "margin", "investment", "company", "finance", "financial", "earnings",
    "assets", "liabilities", "dividend", "equity", "income", "expense"
]

def is_finance_related(text):
    text_lower = text.lower()
    return any(word in text_lower for word in finance_keywords)

def is_definition_query(text):
    text_lower = text.lower().strip()
    triggers = [
        "define", "what is", "what does", "meaning of", "explain term", "definition of"
    ]
    return any(text_lower.startswith(t) for t in triggers) or len(text_lower.split()) <= 3

def chat_fn(user_input):
    if user_input is None or user_input.strip() == "":
        return "❗ Please enter a financial sentence or a financial term."

    if not is_finance_related(user_input):
        return (
            "⚠️ This assistant is specialized in finance.\n\n"
            "Please enter a **finance-related** sentence or term.\n\n"
            "Examples:\n"
            "- profit margin\n"
            "- define revenue\n"
            "- The company reported higher profits due to lower costs."
        )

    user_input = user_input.strip()

    # Decide which instruction to use
    if is_definition_query(user_input):
        # Try to extract the term nicely
        text_lower = user_input.lower()
        for prefix in ["define ", "what is ", "what does ", "definition of ", "meaning of "]:
            if text_lower.startswith(prefix):
                term = user_input[len(prefix):].strip()
                break
        else:
            term = user_input  # e.g., user just typed "profit margin"

        task_type = "📘 Definition"
        prompt = f"<s>[INST] Define the financial term in simple words:\n{term} [/INST]"
    else:
        task_type = "🧾 Explanation"
        prompt = f"<s>[INST] Explain this financial sentence in simple terms:\n{user_input} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    gen_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    # Nicely formatted output
    return f"{task_type}\n\n💬 **Answer:**\n{text}"
    return text
demo = gr.Interface(
    fn=chat_fn,
    inputs=gr.Textbox(
        lines=3,
        label="Enter a finance-related sentence or term",
        placeholder=(
            "Examples:\n"
            "- profit margin\n"
            "- define revenue\n"
            "- The company reported higher profits due to lower costs."
        )
    ),
    outputs=gr.Textbox(label="Assistant Response"),
    title="💰 FinanceExplain Bot",
    description=(
        "This assistant is specialized in **finance**.\n\n"
        "It can:\n"
        "• 📘 Define financial terms (e.g., 'profit margin', 'define revenue')\n"
        "• 🧾 Explain financial sentences in simple words\n\n"
        "Just type a financial term or a finance-related sentence and press Enter."
    ),
    examples=[
        ["profit margin"],
        ["define revenue"],
        ["The company reduced costs and increased profits."],
        ["The firm reported a decline in earnings due to higher expenses."]
    ],
    allow_flagging="never"
)

demo.launch()

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4356c70a1a37a2664b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
test_prompt = "<s>[INST] Explain this financial sentence in simple terms:\nThe company had a favorable profit margin. [/INST]"

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(inputs["input_ids"], max_new_tokens=100)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[INST] Explain this financial sentence in simple terms:
The company had a favorable profit margin. [/INST] This sentence talks about a positive financial situation. In simple words, it means that the company had a favorable profit margin.


## F1 Score
This was computed for the best performing model

In [19]:
import re
from tqdm import tqdm

def generate_preds_refs(model, tokenizer, dataset, n=100):
    preds = []
    refs = []

    for i in tqdm(range(min(n, len(dataset)))):
        text = dataset[i]["text"]

        # Split into instruction and reference
        inst, ref = text.split("[/INST]", 1)
        prompt = inst + "[/INST]"

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=120)

        gen = tokenizer.decode(outputs[0], skip_special_tokens=True)
        gen = gen.replace(prompt, "").strip()

        preds.append(gen)
        refs.append(ref.strip())

    return preds, refs

In [20]:
preds, refs = generate_preds_refs(best_model, tokenizer, val_ds, n=100)

  4%|▍         | 4/100 [00:29<10:52,  6.80s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
  5%|▌         | 5/100 [00:37<11:24,  7.20s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
 19%|█▉        | 19/100 [02:03<07:57,  5.90s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
 20%|██        | 20/100 

In [21]:
from collections import Counter

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

def f1_score_single(pred, ref):
    pred_tokens = normalize(pred)
    ref_tokens = normalize(ref)

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

f1_scores = [f1_score_single(p, r) for p, r in zip(preds, refs)]
avg_f1 = sum(f1_scores) / len(f1_scores)

print("Average F1-score:", avg_f1)

Average F1-score: 0.6756002274665154
